[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/b_13_mini_gpt.ipynb)

# 🔴 Hard: Mini GPT — assemble the whole model

*Attention & Transformers*
Assemble a small LLaMA-style decoder from parts you have already built. Every
piece here is simple; **the marks are all in the wiring**.

### Signature
```python
class MiniGPT(nnx.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, *, rngs): ...
    def __call__(self, ids):    # (B, T) int32 -> (B, T, vocab_size)
```

Expose `self.tok_emb`, `self.blocks` (length `num_layers`), and `self.norm_f`.

### Architecture — modern defaults
- **RMSNorm**, not LayerNorm (`nnx.RMSNorm` is allowed)
- **RoPE** for position — `apply_rope` is given to you in the starter cell
- **SwiGLU** MLP: `w_down(silu(w_gate(x)) * w_up(x))`, hidden width `4 * d_model`
- Pre-norm residuals, exactly as in a GPT-2 block:
  `h = x + Attn(Norm(x))`, then `y = h + MLP(Norm(h))`
- Causal: position $i$ attends to $j \le i$ only
- **Weight tying**: the output head is the token embedding transposed

`nnx.Linear`, `nnx.RMSNorm` and `nnx.Embed` are allowed. `nnx.MultiHeadAttention`
is not — the attention is yours. Use the **provided `apply_rope`** rather than
rolling your own; the point of this problem is where it goes, not what it does.

---

## The four traps

Each one produces a model that runs, trains, and is wrong.

### 1. RoPE goes on `q` and `k`, not on the embedding
The intuition from *learned* positional embeddings — "add position to the input"
— does not transfer. RoPE is applied inside attention, to `q` and `k` only,
after the head split. `v` is never rotated.

Add it to the token embedding instead and the model still runs, still learns
something, and quietly encodes position into the *values* it copies around. The
token embedding must come out of `tok_emb` carrying no positional information at
all; position enters only via attention scores.

### 2. Pre-norm needs a final norm
In pre-norm, each block normalises its **input** and adds an unnormalised
residual. Nothing normalises the output. After `num_layers` blocks the residual
stream has grown, and feeding it straight to the head is a real bug — it is why
every pre-norm model has a `norm_f` between the last block and the logits.

Post-norm architectures don't need one, which is exactly why this gets dropped
when porting.

### 3. The head is the embedding transposed
Tying means the logits are $x E^\top$ using the *same* table you embedded with.
`nnx.Embed.attend(x)` does this. Building a `nnx.Linear(d_model, vocab_size)`
gives an untied model — it runs, but it has `vocab_size × d_model` more
parameters and no longer shares representations between input and output.

At `vocab_size = 50257, d_model = 768` that is 38M parameters — about a third
of GPT-2 small.

### 4. Build the mask once
The causal mask depends only on `T`, so it is the same for every layer. Build it
in `__call__` and pass it down. Rebuilding `jnp.tril(...)` inside each block is
not wrong, just wasteful — and it is the kind of thing an interviewer notices.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


# ---- GIVEN — do not change. This is problem 24, provided so you can focus
# ---- on WHERE it gets applied rather than re-deriving the rotation.
def apply_rope(x, positions):
    """Rotate the feature axis of x by angles set by `positions`.

    Args:
        x:         (..., T, d_head), d_head even
        positions: (T,) integer positions

    Returns:
        Array shaped like x.
    """
    d = x.shape[-1]
    half = d // 2
    inv_freq = 1.0 / (10000.0 ** (jnp.arange(half, dtype=jnp.float32) / half))
    ang = positions[:, None].astype(jnp.float32) * inv_freq[None, :]
    cos, sin = jnp.cos(ang).astype(x.dtype), jnp.sin(ang).astype(x.dtype)
    x1, x2 = x[..., :half], x[..., half:]
    return jnp.concatenate([x1 * cos - x2 * sin, x1 * sin + x2 * cos], axis=-1)


class MiniGPT(nnx.Module):
    """A small LLaMA-style decoder: RMSNorm + RoPE + SwiGLU, weights tied."""

    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, *, rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, ids):
        """(B, T) int32 -> (B, T, vocab_size)"""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

m = MiniGPT(vocab_size=64, d_model=32, num_heads=4, num_layers=3,
            rngs=nnx.Rngs(params=0))
ids = jnp.array([[1, 2, 3, 4, 5]])
print("logits:", m(ids).shape, "-> (B, T, vocab)")

# Trap 1 — position must NOT be in the embedding, but MUST reach the output.
same = jnp.array([[7, 7]])
emb = m.tok_emb(same)
print("\nsame token, two positions — embeddings identical?",
      bool(jnp.allclose(emb[0, 0], emb[0, 1])), "(must be True)")
out = m(same)
print("                            ...but logits differ?",
      not bool(jnp.allclose(out[0, 0], out[0, 1])), "(must be True)")

# Trap 3 — tied head: exactly one vocab-sized matrix in the whole model.
shapes = [v.shape for v in jax.tree.leaves(nnx.state(m))]
print("\nvocab-sized matrices:", sum(1 for s in shapes if 64 in s), "(must be 1)")
print("total params:", sum(int(jnp.prod(jnp.array(s))) for s in shapes))

# Trap 2 — the final norm is really in the path.
before = m(ids)
m.norm_f.scale[...] = m.norm_f.scale[...] * 3.0
print("\nscaling norm_f changed the logits?",
      not bool(jnp.allclose(before, m(ids))), "(must be True)")

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution, status

check("mini_gpt")

# hint("mini_gpt")      # stuck? nudge without the answer
# solution("mini_gpt")  # spoiler: the reference implementation
# status()              # your dashboard across all problems